In [1]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import torch
from peft import PeftModel

In [2]:
def generate_chat(prompt, model, tokenizer):
    messages = [
        {"role": "system", "content": "你是一个十分幽默人，了解很多笑话。直接给出最终笑话，不要输出思考过程。"},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=192,
            do_sample=True,
            temperature=0.9,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    print(tokenizer.decode(new_tokens, skip_special_tokens=True))

# Play with Base Model (Qwen-8B-Inst)

In [3]:
model_name = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        padding_side="right",
    )

model_inst = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="sdpa",
    )
model_inst.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
        (post_attention_la

In [4]:
prompt = "请给我讲一个笑话。"
generate_chat(prompt, model_inst, tokenizer)

有一天，一只鸡走进了一家酒吧，点了一杯啤酒。  
酒保看着它说：“你为什么总是这么晚才来？”  
鸡回答：“因为我得先等我的鸡窝开门！”


# Play with model after SFT 

In [7]:
model_sft = PeftModel.from_pretrained(model_inst, "MRBSTUDIO/Humor-Qwen3-8B-SFT", revision="486c7bf")
model_sft = model_sft.merge_and_unload()
model_sft.eval()

# model_sft_6700 = PeftModel.from_pretrained(model_inst, "MRBSTUDIO/Test-Repo", revision="9a801b0")
# model_sft_6700 = model_sft_6700.merge_and_unload()
# model_sft_6700.eval()

# model_sft_6900 = PeftModel.from_pretrained(model_inst, "MRBSTUDIO/Test-Repo", revision="ab2fbfe")
# model_sft_6900 = model_sft_6900.merge_and_unload()
# model_sft_6900.eval()


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
        (post_attention_la

In [ ]:
prompt = "请介绍一下你自己"
generate_chat(prompt, model_sft, tokenizer)
# generate_chat(prompt, model_sft_6700, tokenizer)
# generate_chat(prompt, model_sft_6900, tokenizer)

今天去一两同学私私私私图一群众脸这里——说个中我完完——图节无节一码样一长腿走路学生，这里一点是一本非常一节一收那样一点两下面一点惊小爱别。#大明星小优。参见。地一优。所寻。——刚刚——百度为小学时：大数）果断一哭。大。控：小大行水一节家团围中团一公。比少一节土里大水睡二治。太子去土团，。小小界水。黄光5。此为一点为量。大救七的个过到量一公大团去一句合节。前者，去解过过小，反。大队。A团。1团数。跨。一点数。方团行？路公。大女。悔来发全。大


# Play with model after GRPO

In [21]:

model_rl = PeftModel.from_pretrained(model_sft, "MRBSTUDIO/Humor-Qwen3-8B-GRPO")
model_rl.eval()

adapter_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/349M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 4096)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [30]:
prompt = "请给我讲一个笑话。"
generate_chat(prompt, model_sft, tokenizer)

我一朋友，有次被老婆打後，哭著求老婆：「妳打我時，為什麼不打我耳光，只打我屁股呢？」     老婆：「因為你打我屁股時，我會痛，我打你耳光時，你會哭。」
